# Day 30 — Customer Churn Prediction 🚀

## Final Project | 30 Days of Machine Learning

Build an end-to-end ML solution to predict whether a telecom customer is likely to churn.

**Workflow:** Data Understanding → Cleaning → EDA → Feature Engineering → Preprocessing → Model Training → Evaluation → Cross-Validation → Feature Importance → Model Persistence → Prediction

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve
)
from xgboost import XGBClassifier

print("Libraries imported successfully!")

## 1. Load the Dataset

In [ ]:
DATA_PATH = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Understand the Dataset

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)
print("\nMissing values:")
display(df.isnull().sum().sort_values(ascending=False).head(10))
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
display(df.describe(include="all").T)

## 3. Data Cleaning

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop_duplicates().reset_index(drop=True)

if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

print("Cleaned shape:", df.shape)
display(df.head())

## 4. Exploratory Data Analysis

In [ ]:
churn_counts = df["Churn"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(["No Churn", "Churn"], churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

display(
    df["Churn"].value_counts(normalize=True)
      .rename(index={0: "No Churn", 1: "Churn"})
      .mul(100).round(2)
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["tenure"].dropna(), bins=20)
plt.title("Customer Tenure Distribution")
plt.xlabel("Tenure (months)")
plt.ylabel("Customers")
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
df["tenure_group"] = pd.cut(
    df["tenure"], bins=[-1, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"]
)

df["monthly_charge_category"] = pd.cut(
    df["MonthlyCharges"], bins=[-np.inf, 35, 70, np.inf],
    labels=["Low", "Medium", "High"]
)

service_columns = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
available = [c for c in service_columns if c in df.columns]
df["service_count"] = sum(
    (df[c].astype(str).str.lower() == "yes").astype(int) for c in available
)

display(df[[
    "tenure", "tenure_group", "MonthlyCharges",
    "monthly_charge_category", "service_count", "Churn"
]].head())

## 6. Prepare Features and Target

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

## 7. Preprocessing Pipeline

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")

## 8. Define Models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=5,
        min_samples_leaf=2, class_weight="balanced",
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.05, max_depth=3,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=250, learning_rate=0.05, max_depth=4,
        min_child_weight=2, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        objective="binary:logistic", eval_metric="logloss",
        random_state=42, n_jobs=-1
    )
}
print("Models:", list(models.keys()))

## 9. Train and Evaluate Models

In [ ]:
trained_pipelines = {}
results = []

for model_name, model in models.items():
    print(f"Training {model_name}...")

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(
    "ROC-AUC", ascending=False
).reset_index(drop=True)

display(results_df.round(4))
results_df.to_csv("Day30_Model_Results.csv", index=False)
print("Saved: Day30_Model_Results.csv")

## 10. Model Comparison

In [ ]:
results_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
].plot(kind="bar", figsize=(12, 6))

plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 11. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = []

for model_name, pipeline in trained_pipelines.items():
    scores = cross_val_score(
        pipeline, X_train, y_train, cv=cv,
        scoring="roc_auc", n_jobs=-1
    )
    cv_results.append({
        "Model": model_name,
        "CV ROC-AUC Mean": scores.mean(),
        "CV ROC-AUC Std": scores.std()
    })

cv_df = pd.DataFrame(cv_results).sort_values(
    "CV ROC-AUC Mean", ascending=False
).reset_index(drop=True)

display(cv_df.round(4))

## 12. Select the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_pipeline = trained_pipelines[best_model_name]

print("Best model based on holdout ROC-AUC:", best_model_name)
print("ROC-AUC:", round(results_df.iloc[0]["ROC-AUC"], 4))

## 13. Classification Report and Confusion Matrix

In [ ]:
best_predictions = best_pipeline.predict(X_test)
best_probabilities = best_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(
    y_test, best_predictions,
    target_names=["No Churn", "Churn"],
    zero_division=0
))

cm = confusion_matrix(y_test, best_predictions)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)
disp.plot()
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 14. ROC Curve

In [ ]:
plt.figure(figsize=(8, 6))

for model_name, pipeline in trained_pipelines.items():
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc_score = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={auc_score:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Feature Importance

In [ ]:
best_estimator = best_pipeline.named_steps["model"]
feature_names = best_pipeline.named_steps["preprocessor"].get_feature_names_out()

if hasattr(best_estimator, "feature_importances_"):
    values = best_estimator.feature_importances_
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": values
    }).sort_values("Importance", ascending=False)

elif hasattr(best_estimator, "coef_"):
    coefficients = best_estimator.coef_[0]
    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importance": np.abs(coefficients),
        "Coefficient": coefficients
    }).sort_values("Importance", ascending=False)

else:
    importance_df = pd.DataFrame()

if not importance_df.empty:
    display(importance_df.head(20))
    importance_df.to_csv("Day30_Feature_Importance.csv", index=False)

    top = importance_df.head(15).sort_values("Importance")
    plt.figure(figsize=(10, 7))
    plt.barh(top["Feature"], top["Importance"])
    plt.title(f"Top Feature Importance — {best_model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Built-in feature importance is not available for this model.")

## 16. Save and Reload the Final Model

In [ ]:
MODEL_FILE = "churn_model.pkl"

joblib.dump(best_pipeline, MODEL_FILE)
print("Saved:", MODEL_FILE)

loaded_model = joblib.load(MODEL_FILE)

sample_customer = X_test.iloc[[0]]
sample_prediction = loaded_model.predict(sample_customer)[0]
sample_probability = loaded_model.predict_proba(sample_customer)[0, 1]

print("\nSample Prediction")
print("Predicted Churn:", "Yes" if sample_prediction == 1 else "No")
print("Churn Probability:", round(sample_probability * 100, 2), "%")

## 17. Business Interpretation 💼

A churn prediction system can help a telecom company identify customers who may leave, prioritize retention campaigns, understand churn-related patterns, and allocate retention resources more efficiently.

**Key point:** Accuracy alone is not enough. Precision, recall, F1-score and ROC-AUC provide a broader view, especially when churn classes are imbalanced.

## 18. Final Takeaways 🎯

1. Real-world datasets need careful cleaning.
2. Feature engineering can add useful predictive information.
3. Pipelines make preprocessing reproducible.
4. Comparing multiple algorithms helps select a suitable model.
5. Cross-validation provides a more reliable performance estimate.
6. ROC-AUC is useful for comparing binary classifiers.
7. Joblib allows the trained pipeline to be reused.
8. The final goal is to convert predictions into useful business decisions.

### Generated Outputs
- `Day30_Model_Results.csv`
- `Day30_Feature_Importance.csv`
- `churn_model.pkl`